![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 05</p>

---

# Visual Object Detection

---
---



# Table of contents for week 05
1. [Introduction to Object Detection](#intro_detec)
2. [Fully Convolutional Networks](#fnc_network)
     1. [Developing an Intuition with VGG16-Feature Extraction](#fnc_intuition)
     2. [The Fully Convolutional Architecture](#fnc_architecture)
3. [Single Shot Detector](#ssd_network)
4. [References](#refs)


## Introduction to Object Detection <a name="intro_detec"></a>
With the success of CNN-based visual object classification after the publication of the Alexnet architecture <a id="anker1" href="#ref1">[1]</a> the use of deep neural networks for the localisation of objects in an image became also a topic of interest. Two simple and intuitive approaches were used at the beginning:

- If by some means object priors were available a standard CNN could be used for their classification. In <a href="#fig1">Fig.1</a> an image segmentation method based on texture, color, and intensity similarity is shown <a id="anker2" href="#ref2">[2]</a>. This method was used for the so-called R-CNN approach (*region proposals with CNNs*), where these priors we classified using a CNN <a id="anker3" href="#ref3">[3]</a> as shown in <a href="#fig2">Fig.2</a>.

<br>
<img src="img/selective_search.png" alt="Drawing" width="700" />
<a id="fig1">Fig.1:</a> Creation of object priors using image segmentation and non-maximum suppression <a id="anker2" href="#ref2">[2]</a>.
<br>

---

<br>
<img src="img/R-CNN.png" alt="Drawing" width="700" />
<a id="fig2">Fig.2:</a> R-CNN approach ("region proposals with CNNs"), where object priors are classified using a CNN <a id="anker3" href="#ref3">[3]</a>.
<br>

---

- The second approach was building upon previous conecepts like *Histogram of Oriented Gradidents* (HOG) <a id="anker4" href="#ref4">[4]</a>, a "pre-CNN" method that built upon manual feature creation and is illustrated in <a href="#fig31">Fig.3.1</a>. The orientation of the edges are used to detect the silhouette of persons (left). The idea is based on the comparison of the edges with a trained template. For the detection in a larger image the template (red) has to be slided over the entire image ideally at different scales (right). This makes the approach costly in terms of processing power. An approach of the same kind is the well-known Viola-Jonas-face detector, which is the algorithm running in real-time on early digital cameras <a id="anker5" href="#ref5">[5]</a>. Here differnet kind of so-called Haar-features (<a href="#fig32">Fig.3.2</a>), which bear similarities to convolutional kernels, are slided over the entire image and strong signals are grouped together to face hypothesis.
- 
Similar concepts were quickly adapted to the CNN-architecture as illustrated in <a href="#fig4">Fig.4</a> <a id="anker6" href="#ref6">[6]</a>. The idea is to use the softmax output of the CNN to create an object score and subsequently a binary mask for each category and finally extract the most relevant detections with a non-maximum suppression.

<br>
<img src="img/HOG.png" alt="Drawing" width="700" />
<a id="fig31">Fig.3.1:</a> Histogram of Oriented Gradidents (HOG) use the orientation of edges to detect the silhouette of persons (left). The detection window of fixed size (red) has to be shifted over the entire image (right) <a id="anker4" href="#ref4">[4]</a>.
<br>

---

<br>
<img src="img/haar_features.png" alt="Drawing" width="550" />
<a id="fig32">Fig.3.2:</a> Five different Haar-features (left) are shown, which are sensitive to certain textures type present in faces <a id="anker5" href="#ref5">[5]</a>.
<br>

---

<br>
<img src="img/DNN_detect.png" alt="Drawing" width="700" />
<a id="fig4">Fig.4:</a> A CNN is slided over an image and binary object masks are created at different scales with a subsequent non-maximum suppression <a id="anker6" href="#ref6">[6]</a>.
<br>

---


## Fully Convolutional Networks <a name="fnc_network"></a>

The idea of Fully Convolutional Networks (FCN) was introduced briefly after the Alexnet in a 2015 <a id="anker7" href="#ref7">[7]</a>. It is used for the purpose of semantic segmentation i.e., the task of classifying every pixel in an image according to the class of the object it belongs to. The important progress was to point out that the dense layers at the top of a CNN could be replaced by convolutional layers to provide as output an object mask for each category. First we want to illustrate the intuitve idea behind this approach using our well know VGG16 architecture.

### Developing an Intuition with VGG16-Feature Extraction <a name="fnc_intuition"></a>

In the iPython notebook on the tansfer learning example [sw04.02.transfer_learn.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.02.transfer_learn.ipynb) we already saw that the feature extraction i.e. convolutional part of a CNN can work on any input image dimension. We used this property to replace the VGG16 classifier by our own - more compact - version. We recall - as shown in the following <a href="#fig5">Fig.5</a> - that with a standard input image size of 224x224 the final feature map is of size 512x7x7. In the following iPython notebook we will illustrate this feature map using maximum or average pooling over all 512 maps but keeping the spatial 7x7 dimension.

<br>
<img src="img/vgg16_features.png" alt="Drawing" width="700" />
<a id="fig5">Fig.5:</a> Feature extraction part of the VGG16 architecture with the final 512x7x7 activation map block</a>.
<br>

---

**Exercise:** 
**[sw05.01.activation_map_live.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.01.activation_map_live.ipynb)**

- Cell [1] - [3]<br>
  These cells should be familiar from the model zoo notebook [sw03.04.model_zoo.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw03.04.model_zoo.ipynb). We download and preapare the `vgg16` architecture.

- Cell [4]<br>
  This cell provides the classification process of an image using the `vgg16` architecture through the function `def classify_image`.
  - Using `weights.transforms` the image is transformed to fit the input requirements of the CNN.
  - The prediction is determined through application of the `vgg16_model` to the image.
  - The relevant object categories i.e., those above the `limit_score` are collected.
  - An output string is constructed with all relevant category labels.

<img src="img/classify_fct.png" alt="Drawing" width="800" />

- Cell [5]<br>
  Here only the feature extraction part of VGG16 is applied to the image and in addition, the pooling layer up to which the propagation shall be performed can be chosen.
  - Using `weights.transforms` the image is transformed to fit the input requirements of the CNN.
  - The image is propagated up to the configured pooling layer (`+1` to include the layer) through application of `vgg16_model.features[:layer+1]` to the image.
  - Then the pooling over the 512 feature maps is done - we only want to show a single output channel - and finally this output is scaled to the standard grayscale image interval $[0,255]$.

<img src="img/activ_map_fct.png" alt="Drawing" width="800" />

- Cell [6]<br>
  This is a helper function used to up-scale the output of the function `activation_map` to the original image dimension be creating a tile-image
  
- Cell [7]<br>
  This cell provides the possibility to apply the above function to a live image grabbed through the standard - if available - video interface of your laptop.
  - At the top all settings for the acquisition and the output format are done.
  - The actuall loop starts with the `while True` statement. It uses the [OpenCV](https://docs.opencv.org/) image acquisiton functionality `cv2.VideoCapture.read()` to acquire images from a video device. Usually the inbuild webcam is at index zero: `capture_device = 0`.
  - We center crop a 448x448 sized region from the image. You can change this value according to the size of the grabbed frame but the input to the CNN will always be 224x224 due to the image transformation. Keep a square-format because `vgg16` seems to center crop non squared images.
  - The image is converted into a `torch.tensor` and the `classify_image` and `activation_map` functions are called. The activation map is scaled up to the original image size and everything is plotted to two windows.
  - You can change the pooling layer index up to which the activation maps are calculated by using the left and right arrow key (Note that the keyboard focus bus be on one of the two output windows). Its value is printed together with the most relevant object categories as overlay on top of the original image.
  - The following figures show respectively (top to bottom) the results of the activation maps after the pooling layers 4, 16 and 30.
  - With the up and down arrow the pooling type over the 512 activation maps can be changed from max- to average-pooling.
  - To **quit** the application push the 'e'-key. Only then the two additional OpenCV windows are correctly closed (`cv2.destroyWindow`).

<img src="img/layer_04.png" alt="Drawing" width="600" />
<img src="img/layer_16.png" alt="Drawing" width="600" />
<img src="img/layer_30.png" alt="Drawing" width="600" />

When observing the activation maps we realise that at each pooling level the strength of the activations - maximum or average - represent some sort of "objectness" in the scene. If in addition we managed to represent this information category-specific we could localise the different objects in the scene. This is actually the basis of the Fully Convolutional Networks. Before we proceed to the FNC, we want to understand a further very important aspect with respect to the application of a CNN feature extaction part to an image. 

The following <a href="#fig6">Fig.6</a> resembles closely <a href="#fig5">Fig.5</a> but with the difference that the VGG16 convolutional part is applied to a 448x448 input image i.e., twice as large in both dimensions as before. This resulting output feature map will be of size 512x14x14 and after pooling over the feature map dimension of 512 the result will be 14x14, which is also twice the size of our previous result. <br>

<br>
<img src="img/vgg16_features_x_2.png" alt="Drawing" width="700" />
<a id="fig6">Fig.6:</a> Feature extraction part of the VGG16 architecture applied to an input of size 448x448, which will produce a final block of activation map of size 512x14x14</a>.
<br>

---

**Exercise:** 
**[sw05.02.activation_map_live_full_size.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.02.activation_map_live_full_size.ipynb)**

This iPython notebook is very similar to the previous one with the exception that in cell [4] an own transformation of the image is defined, which may or may not resize the image to 224x224. In the live view this can be selected with the up and down arrow keys. To visualise the input image difference - 448x448 or 224x224 - we decimate the feature map representation for the 224x224 input size by a factor of two as shown below (left).

<img src="img/half_full_comp.png" alt="Drawing" width="800" />

We now want to compare the propagation of the image size of 448x448 through the `vgg16` to the sliding window approach form <a href="#fig4">Fig.4</a>. We recall that the feature extraction is a subsequent application of convolutional and pooling layers. As illustrated in <a href="#fig7">Fig.7</a> for standard 2x2 pooling to each pixel in the final activation map of size 14x14 corresponds a unique receptive field of 32x32 pixel in the original image (448x448). Furthermore, due to the translational invariance of the convolution its result will be - at each layer - at a given position the same, independant of the fact whether we apply the `vgg16` architecture to the full image or whether we slide it - with 224x224 resolution - over the image. Therefore it turns our that the propagation of the full image through the `vgg16` is equivalent to a sliding `vgg16`-appraoch with a stride of 32. To do so, however, we would have to probe 7x7 i.e. 49 individual positions which represents a tremendous overhead with respect to the "single shot" approach. The reason for this overhead is that for each of the 49 positions a large overlap of the respective images sent through the CNN occurs and convolutions over overlapping parts are calculated multiple times. This important insight, which shows that a sliding window approach of a CNN can be replaced by the application of the CNN to the full image is at the basis of all efficient CCN-based object detection algorithms. In the next section we will show how to create object masks from these activations by replacing the fully connected layers by convolutional layers.

<br>
<img src="img/pooling.png" alt="Drawing" width="700" />
<a id="fig7">Fig.7:</a> For standard 2x2 pooling to each pixel in the final 14x14 activation map a corresponding receptional field of 32x32 pixel in the original image can be associated</a>.
<br>

---

### The Fully Convolutional Architecture <a name="fnc_architecture"></a>

The idea of FCNs was first introduced in a 2015 <a id="anker7" href="#ref7">[7]</a> briefly after the Alexnet for the task of semantic segmentation i.e., of classifying every pixel in an image according to the class of the object it belongs to. In the previous examples we have seen that the feature maps contain the information of an "objectness" for each pixel, which is not too suprsing because the maps represent the max-pooling of all texture information in the image. We now would like to add the information on the individual object class for each pixel. This information is in principle available but "hidden" in the dense layer of the classifier of the CNN. The idea of FCNs is to replace the dense layer by convolutional layers and by this way create a final output of feature maps, where the class identity is coded. We illustrate the idea using our well-known VGG16 architecture, which we recall in the <a href="#fig8">Fig.8</a> below.

<br>
<img src="img/vgg16_segm.png" alt="Drawing" width="700" />
<a id="fig8">Fig.8:</a> The standard VGG16 architecture with the feature extraction and the classification part</a>.
<br>

---

The final output of the feature extraction consists of a 512x7x7 feature map block. Through the flattening this is connected to each of the 4096 neurons of the first dense layer. If we just consider one single neuron we can represent the connection with the features maps as a convolutional kernel of size 512x7x7. If we apply this kernel without padding only one position of can be evaluated, which represents the output of the considered dense neuron. Extending this idea to all 4096 dense neurons we realise that the first fully connected layer can be replaced by a convolutional layer with a set of 4096 kernels of size 512x7x7 applied without padding. The result will be a 4096x1x1 feature map corresponding to the full output of the first dense layer. The same idea applied to the two additional dense layers, which can be replaced by a set of 4096 and 1000 convolutional kernels of size 4096x1x1 respectively we end up with the architecture represented in <a href="#fig9">Fig.9</a>. The final output feature map of size 1000x1x1 corresponds to the class probabilities of the 1000 categories for which the model was trained for.

<br>
<img src="img/vgg_fcn.png" alt="Drawing" width="700" />
<a id="fig9">Fig.9:</a> The standard VGG16 architecture with the the classification part represented using convolutional layers</a>.
<br>

---

While up to this point we just replaced the dense layers by convolutions we can now apply the idea of the previous chapter and apply this network architecture to larger images. Because it is a fully convolutional network it does not require a defined input image size but will produce a coarse grained output view of the scene (decimated by 32 in x- and y-direction) with an object probability for each output pixel. This is in fact the main idea of FCNs. <br>
The architecture of the original publication <a id="anker7" href="#ref7">[7]</a>  was based on the AlexNet backbone and is represented in <a href="#fig10">Fig.10</a>. Note the three final feature maps in the classifier of sizes 4096x1x1 (twice) and 21x1x1 which correspond to our VGG16-based illustration in <a href="#fig9">Fig.9</a>, but with 21 output categories ('0' being the background) only. As a final step the raw output of the classifier is upsampled (using linear interpolation) to the size of the original image.

<br>
<img src="img/fcn_alex.png" alt="Drawing" width="550" />
<a id="fig10">Fig.10:</a> The original FNC-architecture <a id="anker7" href="#ref7">[7]</a> based on the AlexNet backbone</a>.
<br>

---

**Exercise:** 
**[sw05.03.fcn_live.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.03.fcn_live.ipynb)**

- Cell [1] - [7]<br>
  These cells should be familiar from the model zoo notebook [sw03.04.model_zoo.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw03.04.model_zoo.ipynb). We download and preapare the `fcn_resnet50` architecture. You can observe that the backbone of `fcn_resnet50` is not the VGG16 net but the ResNet50 <a id="anker8" href="#ref8">[8]</a>. This represents an update of the implementation with respect to the original implementation <a id="anker7" href="#ref7">[7]</a>, which was based on the AlexNet architecture. <br>
  You can identify the `FCNHead`, which consists of two convolutional layers of sizes 512x3x3 (2048 kernels) and 21x1x1 (512 kernels) which also represents a change with respect to the original paper.

- Cell [8]<br>
  Here we apply the backbone and subsequently the classifier to the transformed image and print the corresponding output sizes. The transformed input image of size 520x809 is decimated by a factor of 8 in the backbone to an output of 65x102. The classifier does not change the size and the final feature map is of size 21x65x102, with the 21 channels corresponding to the different object categories. We expect category ID 8 (=cat) and plot the corresponding feature map, which represents well the coarse-grained (through the decimation) pixel corresponding to the original cat.

<img src="img/cat_mask.png" alt="Drawing" width="160" />

- Cell [9] and [10]<br>
  He we apply the full `fcn_resnet50` architecture including the upsampling to the original image size and extract the contour of the cat using standard image processing techniques from [OpenCV](https://docs.opencv.org/) library.

<img src="img/cat_contour.png" alt="Drawing" width="320" />

- Cell [11]<br>
  This cell provides - similar to the previous iPython notebooks - the possibility to apply the FNC concept to a live stream of your webcam.

Based on the understanding of the FNC network basic the step to the bouding box detectors like SSD or YOLO is quite small. We use as an illustrative example the SSD architecture because it is immediatly available in the torchvision library. In addition, an iPython notebook explaining the use of YOLOv8 is given because of the high relevance of this detector in practice.

### Single Shot Detector <a name="ssd_network"></a>

The Single Shot Detector (SSD) <a id="anker9" href="#ref9">[9]</a> makes use of the above ideas for a detection not at a pixel level (semantic segmentation) but at object level trough the prediction of bouding boxes and corresponding class IDs. At each position and for six different scales a given set of default boxes (4 or 6) are used and the shape offset - with respect to the default boxes - and confidence for each class ID is determined <a href="#fig111">Fig.11.1</a>.<br>
The backbone architecture is based on the VGG16 <a href="#fig112">Fig.11.2</a>, up to the end of the forth convolutional layer block i.e. just before the forth pooling layer. The input image size is scaled to 300x300, which produces feature map outputs of size 512x38x38. The additional convolutional blocks represent the prediciton of bounding box shape offsets and class labels at six different scales. The corresponding kernel sizes are `3x3x[k x (Classes + 4)]` with k being equal to the number of default boxes (4 or 6 depending on the scale) and `Classes` represents the number of output classes. Finally a non-maximum suppression is applied to select the most relevant boxes and corresponding categories.

<br>
<img src="img/ssd_default_box.png" alt="Drawing" width="750" />
<a id="fig111">Fig.11.1:</a> SSD uses a given number of default boxes (here 4) at different scales to predict object positions (<a id="anker9" href="#ref9">[9]</a>)</a>.
<br>

---

<br>
<img src="img/ssd_architecture.png" alt="Drawing" width="850" />
<a id="fig112">Fig.11.2:</a> The SSD-architecture <a id="anker9" href="#ref9">[9]</a> based on the VGG16 backbone</a>.
<br>

---


**Exercise:** 
**[sw05.04.ssd_live.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.04.ssd_live.ipynb)**

The SSD-architecture is slightly more complex than FCN due to the bounding box predicitons at different scales. Therefore we only have a look at a high level on the implementation. Details can be seen in the source code ([Link](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/ssd.py)).

- Cell [1] - [7]<br>
  This part is very similar to the script above [sw05.03.fcn_live.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.03.fcn_live.ipynb). We download and preapare the `ssd300_vgg16` architecture. You can observe the `feature` part in the backbone corresponding to VGG16 till layer index 20. The `extra` part in the backbone are the additional feature layers, which predict boxes and class IDs at different scales.<br>
  You can identify the `FCNHead`, which consists of the `classification_head`, for the class ID predictions, and the `regression_head`, for the bouding box estimation.

- Cell [8] - [9]<br>
  Here the classification is done and - for reasons of simplicity - only the highest class score is extracted. In cell [9] the correspondig bounding box is extracted and plotted as overlay on the image.

<img src="img/cat_box.png" alt="Drawing" width="320" />

- Cell [10]<br>
  As for the previous exercise the possibility for a live processing is given with output of the most relevant bouding boxes and classes as overlay on the image. The processing time is considerably smaller than for the FCN network due to the smaller input image size and the simplified backbone (VGG16 instead of ResNet50).


As a final example the YOLO detector <a id="anker10" href="#ref10">[10]</a> can be explored with the following iPython notebook.

**Exercise:** 
**[sw05.05.yolo.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.05.yolo.ipynb)**


## References <a name="refs"></a>

<a id="ref1" href="#anker1">[1]</a> Krizhevsky, Alex, Sutskever, Ilya, and Hinton, Geoffrey E. Imagenet classification with deep convolutional neural networks. In Advances in neural information processing systems, pp. 1097–1105,
2012.

<a id="ref2" href="#anker2">[2]</a> Uijlings, Jasper RR, et al. "Selective search for object recognition." International journal of computer vision 104 (2013): 154-171.

<a id="ref3" href="#anker3">[3]</a> Girshick, Ross, et al. Rich feature hierarchies for accurate object detection and semantic segmentation. Proceedings of the IEEE conference on computer vision and pattern recognition. 2014.

<a id="ref4" href="#anker4">[4]</a> Dalal, Navneet, and Bill Triggs. "Histograms of oriented gradients for human detection." 2005 IEEE computer society conference on computer vision and pattern recognition (CVPR'05). Vol. 1. Ieee, 2005.

<a id="ref5" href="#anker5">[5]</a> Viola, Paul, and Michael Jones. "Rapid object detection using a boosted cascade of simple features." Proceedings of the 2001 IEEE computer society conference on computer vision and pattern recognition. CVPR 2001. Vol. 1. Ieee, 2001.

<a id="ref6" href="#anker6">[6]</a> Szegedy, Christian, Alexander Toshev, and Dumitru Erhan. "Deep neural networks for object detection." Advances in neural information processing systems 26 (2013).

<a id="ref7" href="#anker7">[7]</a> Long, Jonathan, Evan Shelhamer, and Trevor Darrell. "Fully convolutional networks for semantic segmentation." Proceedings of the IEEE conference on computer vision and pattern recognition. 2015.

<a id="ref8" href="#anker8">[8]</a> He, Kaiming, et al. "Deep residual learning for image recognition." Proceedings of the IEEE conference on computer vision and pattern recognition. 2016.

<a id="ref9" href="#anker9">[9]</a> Liu, Wei, et al. "Ssd: Single shot multibox detector." Computer Vision–ECCV 2016: 14th European Conference, Amsterdam, The Netherlands, October 11–14, 2016, Proceedings, Part I 14. Springer International Publishing, 2016.

<a id="ref10" href="#anker10">[10]</a> Redmon, J., Divvala, S., Girshick, R., Farhadi, A.: You only look once: Unified, real-time
object detection. In: CVPR. (2016)